# 06 — Export một MRI study mẫu cho Local Demo

Notebook này **chỉ chuẩn bị dữ liệu mẫu**, không train, không load model và không tạo submission.
Notebook chạy trên Kaggle vì toàn bộ DICOM của competition vẫn nằm ở đó.

## Notebook làm gì?

1. Đọc `train_series.csv` và `shared_split.csv`.
2. Ưu tiên chọn một study trong `gold_validation` của shared split.
3. Chạy lại đúng quy tắc chọn series đang dùng trong notebook ensemble cuối:
   - **ViT-B/16:** chọn một series cho mỗi plane (Sagittal, Coronal, Axial).
   - **DINOv2:** chọn năm role `SAG_FLUID`, `COR_FLUID`, `AX_FLUID`, `SAG_NONFLUID`, `COR_NONFLUID`.
4. Lấy **hợp các series** được hai model chọn, không tải toàn bộ study và không tải toàn bộ competition.
5. Copy toàn bộ DICOM slices thuộc các series được chọn, kèm metadata, selection log và manifest vào `demo_sample.zip`.

## Kaggle Inputs cần attach

- Competition: **RSNA Knee Abnormality Detection** (có `train_series.csv` và thư mục `train_images/` hoặc tương đương).
- Dataset artifact của ViT cuối: cần `shared_split.csv` (notebook đọc được file đã giải nén hoặc đọc trực tiếp từ `vit_pretrained_centerprior_model_artifacts.zip`).

**Không cần attach pseudo-label batches, không cần attach checkpoint DINOv2/ViT cho notebook export này.**

> Study được chọn từ tập `gold_validation` của shared split. Dữ liệu chỉ dùng để trình diễn local; không dùng prediction của study này để tuyên bố đánh giá độc lập. DINOv2 ensemble gồm 5 fold, trong đó một số fold có thể đã nhìn thấy study này trong training.

## 1. Imports và cấu hình

Chỉ cần chạy CPU. Không cần bật GPU và không cần Internet.

In [ ]:
from __future__ import annotations

import os
import json
import zipfile
import warnings
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pydicom
from IPython.display import display, FileLink

warnings.filterwarnings("ignore", category=UserWarning)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

# Để trống để tự dò. Chỉ sửa nếu Kaggle mount dữ liệu ở vị trí khác.
COMP_ROOT_OVERRIDE = ""             # Thư mục chứa train_series.csv
TRAIN_DICOM_ROOT_OVERRIDE = ""      # Thư mục chứa StudyUID/SeriesUID/*.dcm
SHARED_SPLIT_CSV_OVERRIDE = ""      # Đường dẫn đến shared_split.csv

# Để trống: tự chọn study phù hợp nhất trong gold_validation.
# Có thể điền StudyInstanceUID cụ thể sau khi xem bảng candidate.
SAMPLE_STUDY_UID = ""
SAMPLE_SPLIT_VALUE = "gold_validation"

# Study cần có ít nhất số role/plane này mới được xem là candidate.
# Ưu tiên 5/5 DINO roles và 3/3 ViT planes; có thể hạ ngưỡng nếu cần.
MIN_DINO_ROLES = 3
MIN_VIT_PLANES = 3

# Số DICOM tối thiểu trong mỗi series để cả hai preprocessing pipeline
# có đủ cơ hội chọn ít nhất 4 slices decode được.
MIN_DICOM_FILES_PER_SERIES = 4

# Giới hạn kích thước để tránh vô tình export quá nhiều dữ liệu.
# Tăng giới hạn nếu một study hợp lệ lớn hơn mức này.
MAX_EXPORT_GB = 2.0
PREFLIGHT_PIXEL_DECODE = True  # Decode thử 1 slice/series trước khi copy

EXPORT_DIR = WORK_ROOT / "rsna_demo_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ZIP = EXPORT_DIR / "demo_sample.zip"

ID_COL = "StudyInstanceUID"
SERIES_COL = "SeriesInstanceUID"
REQUIRED_SERIES_COLUMNS = {
    ID_COL, SERIES_COL, "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"
}

VIT_PLANE_ORDER = ["Sagittal", "Coronal", "Axial"]
DINO_ROLE_ORDER = ["SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"]
DINO_ROLE_SPECS = {
    "SAG_FLUID": ("sagittal", 1),
    "COR_FLUID": ("coronal", 1),
    "AX_FLUID": ("axial", 1),
    "SAG_NONFLUID": ("sagittal", 0),
    "COR_NONFLUID": ("coronal", 0),
}

print("Input root:", INPUT_ROOT)
print("pydicom:", pydicom.__version__)
print("Output:", OUTPUT_ZIP)

## 2. Tìm competition data, DICOM root và shared split

In [ ]:
SKIP_SEARCH_DIRS = {
    "train_images", "test_images", "train_series", "test_series",
    "train_dicom", "test_dicom", "dicom", ".git", "__pycache__"
}


def find_named_files(root: Path, filename: str) -> list[Path]:
    """Find files by exact basename without descending into DICOM image trees."""
    root = Path(root)
    if not root.exists():
        return []
    found = {}
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d.lower() not in SKIP_SEARCH_DIRS]
        if filename in files:
            path = Path(current) / filename
            found[str(path.resolve())] = path
    return list(found.values())


def discover_competition_root() -> Path:
    if COMP_ROOT_OVERRIDE:
        root = Path(COMP_ROOT_OVERRIDE).expanduser()
        if not (root / "train_series.csv").is_file():
            raise FileNotFoundError(f"Không tìm thấy train_series.csv trong COMP_ROOT_OVERRIDE: {root}")
        return root

    direct = INPUT_ROOT / "competitions"
    candidates = []
    if direct.is_dir():
        candidates.extend(p for p in direct.iterdir() if (p / "train_series.csv").is_file())
    if not candidates:
        candidates = [p.parent for p in find_named_files(INPUT_ROOT, "train_series.csv")]
    unique = {str(p.resolve()): p for p in candidates}
    if len(unique) != 1:
        raise RuntimeError(
            "Không xác định được duy nhất competition root. "
            f"Candidates: {[str(x) for x in unique.values()]}. "
            "Hãy đặt COMP_ROOT_OVERRIDE ở Cell 1."
        )
    return next(iter(unique.values()))


COMP_ROOT = discover_competition_root()
TRAIN_SERIES_CSV = COMP_ROOT / "train_series.csv"

series_root_candidates = [
    COMP_ROOT / "train_images",
    COMP_ROOT / "train_series",
    COMP_ROOT / "train_dicom",
    COMP_ROOT / "train",
]
if TRAIN_DICOM_ROOT_OVERRIDE:
    TRAIN_DICOM_ROOT = Path(TRAIN_DICOM_ROOT_OVERRIDE).expanduser()
else:
    TRAIN_DICOM_ROOT = next((p for p in series_root_candidates if p.is_dir()), None)
    if TRAIN_DICOM_ROOT is None:
        raise FileNotFoundError(
            "Không tìm thấy thư mục DICOM train. Đã kiểm tra train_images/, train_series/, "
            "train_dicom/, train/. Hãy đặt TRAIN_DICOM_ROOT_OVERRIDE."
        )

SHARED_SPLIT_CSV = None
SHARED_SPLIT_ZIP_PATH = None
SHARED_SPLIT_ZIP_MEMBER = None
if SHARED_SPLIT_CSV_OVERRIDE:
    SHARED_SPLIT_CSV = Path(SHARED_SPLIT_CSV_OVERRIDE).expanduser()
    if not SHARED_SPLIT_CSV.is_file():
        raise FileNotFoundError(SHARED_SPLIT_CSV)
else:
    split_candidates = find_named_files(INPUT_ROOT, "shared_split.csv")
    # Prefer the split packaged alongside the final pretrained ViT artifact.
    preferred = [p for p in split_candidates if "vit_pretrained" in str(p).lower() or "vit-pretrained" in str(p).lower()]
    if len(preferred) == 1:
        SHARED_SPLIT_CSV = preferred[0]
    elif len(split_candidates) == 1:
        SHARED_SPLIT_CSV = split_candidates[0]
    elif not split_candidates:
        # If Kaggle Input contains the artifact ZIP rather than an extracted folder,
        # read shared_split.csv from inside the ZIP without unpacking model weights.
        zip_candidates = find_named_files(INPUT_ROOT, "vit_pretrained_centerprior_model_artifacts.zip")
        if len(zip_candidates) == 1:
            SHARED_SPLIT_ZIP_PATH = zip_candidates[0]
            with zipfile.ZipFile(SHARED_SPLIT_ZIP_PATH, "r") as split_zip:
                members = [name for name in split_zip.namelist() if Path(name).name == "shared_split.csv"]
            if len(members) != 1:
                raise RuntimeError(
                    "Không tìm thấy duy nhất shared_split.csv trong ViT artifact ZIP: "
                    f"{members}"
                )
            SHARED_SPLIT_ZIP_MEMBER = members[0]
        else:
            raise RuntimeError(
                "Không tìm thấy shared_split.csv đã giải nén hoặc đúng một "
                "vit_pretrained_centerprior_model_artifacts.zip. "
                "Hãy attach Dataset artifact ViT cuối hoặc đặt SHARED_SPLIT_CSV_OVERRIDE. "
                f"ZIP candidates: {[str(p) for p in zip_candidates]}"
            )
    else:
        raise RuntimeError(
            "Tìm thấy nhiều shared_split.csv nhưng không xác định được file của ViT cuối. "
            "Hãy đặt SHARED_SPLIT_CSV_OVERRIDE. "
            f"Candidates: {[str(p) for p in split_candidates]}"
        )

print("Competition root:", COMP_ROOT)
print("Train DICOM root:", TRAIN_DICOM_ROOT)
print("Shared split:", SHARED_SPLIT_CSV or f"{SHARED_SPLIT_ZIP_PATH} :: {SHARED_SPLIT_ZIP_MEMBER}")

## 3. Đọc metadata và dựng lại quy tắc chọn series của hai model

Đoạn code này bám theo selection logic trong notebook ensemble cuối. Không chọn series bằng tên thư mục hoặc đoán từ SeriesDescription.

In [ ]:
series_meta = pd.read_csv(TRAIN_SERIES_CSV, dtype={ID_COL: str, SERIES_COL: str})
missing = REQUIRED_SERIES_COLUMNS - set(series_meta.columns)
if missing:
    raise ValueError(f"train_series.csv thiếu các cột: {sorted(missing)}")

series_meta[ID_COL] = series_meta[ID_COL].astype(str)
series_meta[SERIES_COL] = series_meta[SERIES_COL].astype(str)


def normalize_plane(value: object) -> str:
    text = str(value).strip().lower()
    if "sag" in text:
        return "sagittal"
    if "cor" in text:
        return "coronal"
    if "axi" in text or "trans" in text or text == "ax":
        return "axial"
    return text


def parse_binary_flag(value: object) -> int:
    if pd.isna(value):
        return -1
    text = str(value).strip().lower()
    mapping = {
        "true": 1, "false": 0, "yes": 1, "no": 0,
        "y": 1, "n": 0, "1": 1, "0": 0,
        "1.0": 1, "0.0": 0,
    }
    if text in mapping:
        return mapping[text]
    try:
        number = float(text)
        return int(number) if number in (0.0, 1.0) else -1
    except (TypeError, ValueError):
        return -1


# Separate normalization columns to mirror the final inference notebook exactly:
# DINOv2 accepts axial/transverse aliases; ViT uses title-cased exact plane names.
series_meta["_plane_norm"] = series_meta["Anatomical_Plane"].map(normalize_plane)
series_meta["_vit_plane_norm"] = series_meta["Anatomical_Plane"].astype(str).str.strip().str.title()
series_meta["_fluid_num"] = series_meta["Fluid_Sensitive"].map(parse_binary_flag).astype(int)
series_meta["_fat_num"] = series_meta["Fat_Suppression"].map(parse_binary_flag).astype(int)

if series_meta[SERIES_COL].duplicated().any():
    duplicates = series_meta.loc[series_meta[SERIES_COL].duplicated(), SERIES_COL].head().tolist()
    raise ValueError(f"SeriesInstanceUID bị trùng trong train_series.csv: {duplicates}")


def choose_primary_per_plane(group: pd.DataFrame) -> dict[str, pd.Series | None]:
    primary = {}
    for plane in ("sagittal", "coronal", "axial"):
        candidates = group[group["_plane_norm"].eq(plane)].copy()
        if candidates.empty:
            primary[plane] = None
            continue
        candidates = candidates.sort_values(
            ["_fluid_num", "_fat_num", SERIES_COL],
            ascending=[False, False, True], kind="mergesort"
        )
        primary[plane] = candidates.iloc[0]
    return primary


def select_both_models(group: pd.DataFrame) -> dict:
    """Return exact selected series for final ViT and DINOv2 inference rules."""
    group = group.copy()
    primary = choose_primary_per_plane(group)

    # ViT-B/16: one highest-priority series for each anatomical plane.
    vit = {}
    for plane in VIT_PLANE_ORDER:
        candidates = group[group["_vit_plane_norm"].eq(plane)].copy()
        if candidates.empty:
            vit[plane] = None
            continue
        candidates = candidates.sort_values(
            ["_fluid_num", "_fat_num", SERIES_COL],
            ascending=[False, False, True], kind="mergesort"
        )
        vit[plane] = str(candidates.iloc[0][SERIES_COL])

    # DINOv2: primary-per-plane rule, then exact-fluid fallback ranked by fat suppression.
    dino = {}
    dino_sources = {}
    for role in DINO_ROLE_ORDER:
        plane, fluid_flag = DINO_ROLE_SPECS[role]
        candidates = group[
            group["_plane_norm"].eq(plane) & group["_fluid_num"].eq(fluid_flag)
        ].copy()
        primary_row = primary.get(plane)
        selected_uid = None
        selection_source = "missing"

        if primary_row is not None and int(primary_row["_fluid_num"]) == fluid_flag:
            selected_uid = str(primary_row[SERIES_COL])
            selection_source = "primary_cache_equivalent"
        elif not candidates.empty:
            candidates["_fat_rank"] = candidates["_fat_num"].map({0: 0, -1: 1, 1: 2}).fillna(1)
            candidates = candidates.sort_values(
                ["_fat_rank", SERIES_COL], ascending=[True, True], kind="mergesort"
            )
            selected_uid = str(candidates.iloc[0][SERIES_COL])
            selection_source = "supplementary_fallback"

        dino[role] = selected_uid
        dino_sources[role] = selection_source

    return {
        "vit": vit,
        "dino": dino,
        "dino_sources": dino_sources,
    }

print("Studies in train_series.csv:", series_meta[ID_COL].nunique())
print("Series rows:", len(series_meta))
print("Metadata columns:", list(series_meta.columns))

## 4. Lọc study trong gold validation và tìm study có dữ liệu DICOM đầy đủ

In [ ]:
if SHARED_SPLIT_CSV is not None:
    shared_split = pd.read_csv(SHARED_SPLIT_CSV, dtype={ID_COL: str})
else:
    with zipfile.ZipFile(SHARED_SPLIT_ZIP_PATH, "r") as split_zip:
        with split_zip.open(SHARED_SPLIT_ZIP_MEMBER, "r") as split_stream:
            shared_split = pd.read_csv(split_stream, dtype={ID_COL: str})
if ID_COL not in shared_split.columns or "split" not in shared_split.columns:
    raise ValueError("shared_split.csv cần có StudyInstanceUID và split.")
shared_split[ID_COL] = shared_split[ID_COL].astype(str)
shared_split["split"] = shared_split["split"].astype(str).str.strip()
if shared_split[ID_COL].duplicated().any():
    raise ValueError("StudyInstanceUID bị trùng trong shared_split.csv")

print("Split counts:")
display(shared_split["split"].value_counts().rename_axis("split").reset_index(name="studies"))

validation_uids = shared_split.loc[
    shared_split["split"].eq(SAMPLE_SPLIT_VALUE), ID_COL
].astype(str).tolist()
if not validation_uids:
    raise ValueError(
        f"Không có study nào trong split='{SAMPLE_SPLIT_VALUE}'. "
        f"Các giá trị hiện có: {sorted(shared_split['split'].unique().tolist())}"
    )

if SAMPLE_STUDY_UID:
    requested_uid = str(SAMPLE_STUDY_UID).strip()
    if requested_uid not in set(validation_uids):
        raise ValueError(
            f"SAMPLE_STUDY_UID={requested_uid} không thuộc split {SAMPLE_SPLIT_VALUE}. "
            "Để giữ đúng validation sample, hãy chọn UID trong bảng candidate."
        )
    candidate_uids = [requested_uid]
else:
    candidate_uids = validation_uids

series_groups = {
    str(uid): group.copy()
    for uid, group in series_meta.groupby(ID_COL, sort=False)
}

candidate_rows = []
selection_by_study = {}
for uid in candidate_uids:
    group = series_groups.get(uid)
    if group is None or group.empty:
        continue
    selection = select_both_models(group)
    dino_count = sum(value is not None for value in selection["dino"].values())
    vit_count = sum(value is not None for value in selection["vit"].values())
    selected_series = sorted({
        value for value in list(selection["dino"].values()) + list(selection["vit"].values())
        if value is not None
    })
    candidate_rows.append({
        ID_COL: uid,
        "dino_roles_selected": dino_count,
        "dino_roles_total": len(DINO_ROLE_ORDER),
        "vit_planes_selected": vit_count,
        "vit_planes_total": len(VIT_PLANE_ORDER),
        "n_unique_series_to_export": len(selected_series),
    })
    selection_by_study[uid] = selection

candidate_df = pd.DataFrame(candidate_rows)
if candidate_df.empty:
    raise RuntimeError("Không tìm được candidate trong shared split và train_series.csv.")

candidate_df = candidate_df.sort_values(
    ["dino_roles_selected", "vit_planes_selected", "n_unique_series_to_export", ID_COL],
    ascending=[False, False, True, True], kind="mergesort"
).reset_index(drop=True)
print("Candidate studies:", len(candidate_df))
display(candidate_df.head(20))

if not SAMPLE_STUDY_UID:
    best_meta = candidate_df[
        (candidate_df["dino_roles_selected"] >= MIN_DINO_ROLES)
        & (candidate_df["vit_planes_selected"] >= MIN_VIT_PLANES)
    ]
    if best_meta.empty:
        raise RuntimeError(
            "Không có study validation đạt ngưỡng role/plane đã đặt. "
            "Hãy giảm MIN_DINO_ROLES hoặc MIN_VIT_PLANES ở Cell 1."
        )
    candidate_uids_to_check = best_meta[ID_COL].astype(str).tolist()
else:
    candidate_uids_to_check = [str(SAMPLE_STUDY_UID).strip()]


def list_series_dicoms(study_uid: str, series_uid: str) -> list[Path]:
    folder = TRAIN_DICOM_ROOT / str(study_uid) / str(series_uid)
    if not folder.is_dir():
        return []
    # Both training and final inference pipelines consume .dcm files.
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".dcm")


def preflight_series(study_uid: str, series_uid: str, files: list[Path]) -> None:
    if len(files) < MIN_DICOM_FILES_PER_SERIES:
        raise RuntimeError(
            f"Series {series_uid} chỉ có {len(files)} DICOM; cần >= {MIN_DICOM_FILES_PER_SERIES}."
        )
    probe = files[len(files) // 2]
    header = pydicom.dcmread(
        str(probe), stop_before_pixels=True, force=True,
        specific_tags=["StudyInstanceUID", "SeriesInstanceUID"]
    )
    actual_study = str(getattr(header, "StudyInstanceUID", ""))
    actual_series = str(getattr(header, "SeriesInstanceUID", ""))
    if actual_study and actual_study != str(study_uid):
        raise RuntimeError(f"DICOM StudyInstanceUID mismatch: {actual_study} != {study_uid}")
    if actual_series and actual_series != str(series_uid):
        raise RuntimeError(f"DICOM SeriesInstanceUID mismatch: {actual_series} != {series_uid}")
    if PREFLIGHT_PIXEL_DECODE:
        decoded = pydicom.dcmread(str(probe), force=True)
        pixels = np.asarray(decoded.pixel_array)
        if pixels.size == 0:
            raise RuntimeError(f"Pixel array rỗng: {probe.name}")


chosen_uid = None
chosen_selection = None
chosen_files_by_series = None
candidate_failures = []

for uid in candidate_uids_to_check:
    selection = selection_by_study[uid]
    if sum(v is not None for v in selection["dino"].values()) < MIN_DINO_ROLES:
        continue
    if sum(v is not None for v in selection["vit"].values()) < MIN_VIT_PLANES:
        continue

    series_uids = sorted({
        value for value in list(selection["dino"].values()) + list(selection["vit"].values())
        if value is not None
    })
    files_by_series = {}
    try:
        total_bytes = 0
        for series_uid in series_uids:
            files = list_series_dicoms(uid, series_uid)
            preflight_series(uid, series_uid, files)
            files_by_series[series_uid] = files
            total_bytes += sum(p.stat().st_size for p in files)
        if total_bytes > MAX_EXPORT_GB * (1024 ** 3):
            raise RuntimeError(
                f"Tổng DICOM {total_bytes / (1024 ** 3):.2f} GiB vượt MAX_EXPORT_GB={MAX_EXPORT_GB}."
            )
        chosen_uid = uid
        chosen_selection = selection
        chosen_files_by_series = files_by_series
        break
    except Exception as exc:
        candidate_failures.append({ID_COL: uid, "reason": f"{type(exc).__name__}: {exc}"})

if chosen_uid is None:
    display(pd.DataFrame(candidate_failures))
    raise RuntimeError(
        "Không tìm được study DICOM hợp lệ trong candidate list. "
        "Kiểm tra DICOM root, codec hoặc giảm MIN_DINO_ROLES / MIN_VIT_PLANES / tăng MAX_EXPORT_GB."
    )

print("Selected study:", chosen_uid)
print("Selected DINO roles:", sum(v is not None for v in chosen_selection["dino"].values()), "/ 5")
print("Selected ViT planes:", sum(v is not None for v in chosen_selection["vit"].values()), "/ 3")
print("Unique selected series:", len(chosen_files_by_series))
print("Pixel decode preflight:", "PASS" if PREFLIGHT_PIXEL_DECODE else "SKIPPED")

## 5. Xem selection cuối cùng trước khi export

In [ ]:
selection_rows = []
for role in DINO_ROLE_ORDER:
    series_uid = chosen_selection["dino"].get(role)
    selection_rows.append({
        "model": "DINOv2",
        "role_or_plane": role,
        SERIES_COL: series_uid or "",
        "selection_source": chosen_selection["dino_sources"].get(role, "missing"),
    })
for plane in VIT_PLANE_ORDER:
    series_uid = chosen_selection["vit"].get(plane)
    selection_rows.append({
        "model": "ViT-B/16",
        "role_or_plane": plane,
        SERIES_COL: series_uid or "",
        "selection_source": "highest_fluid_fat_priority_per_plane" if series_uid else "missing",
    })

selection_df = pd.DataFrame(selection_rows)
display(selection_df)

selected_for = defaultdict(list)
for role, series_uid in chosen_selection["dino"].items():
    if series_uid:
        selected_for[series_uid].append(f"DINOv2:{role}")
for plane, series_uid in chosen_selection["vit"].items():
    if series_uid:
        selected_for[series_uid].append(f"ViT:{plane}")

selected_uid_set = set(chosen_files_by_series)
selected_metadata = series_meta[
    series_meta[ID_COL].eq(chosen_uid) & series_meta[SERIES_COL].isin(selected_uid_set)
].copy()
selected_metadata["selected_for"] = selected_metadata[SERIES_COL].map(
    lambda value: ";".join(selected_for[str(value)])
)
selected_metadata["dicom_relative_dir"] = selected_metadata.apply(
    lambda row: f"dicom/{chosen_uid}/{row[SERIES_COL]}", axis=1
)

# Ensure every selected series is represented once in metadata.
if set(selected_metadata[SERIES_COL].astype(str)) != selected_uid_set:
    raise RuntimeError("Selected metadata rows do not match exported DICOM series.")
if selected_metadata[SERIES_COL].duplicated().any():
    raise RuntimeError("Selected metadata contains duplicate SeriesInstanceUID rows.")

series_size_rows = []
for series_uid, files in chosen_files_by_series.items():
    series_size_rows.append({
        SERIES_COL: series_uid,
        "selected_for": ";".join(selected_for[series_uid]),
        "n_dicom_files": len(files),
        "size_mb": round(sum(p.stat().st_size for p in files) / (1024 ** 2), 2),
    })
series_size_df = pd.DataFrame(series_size_rows).sort_values(SERIES_COL)
display(series_size_df)
print("Total files:", int(series_size_df["n_dicom_files"].sum()))
print("Total size (MiB):", round(float(series_size_df["size_mb"].sum()), 2))

## 6. Đóng gói `demo_sample.zip`

ZIP dùng `ZIP_STORED` để tránh nén lại pixel data vốn thường đã được mã hóa trong DICOM. Notebook copy toàn bộ slices của các series được chọn, nhưng không copy các series khác hoặc nhãn training.

In [ ]:
created_at = datetime.now(timezone.utc).isoformat()
source_split = SAMPLE_SPLIT_VALUE

manifest = {
    "format": "rsna-knee-local-demo-sample",
    "format_version": 1,
    "created_at_utc": created_at,
    "source": {
        "competition": "RSNA Knee Abnormality Detection",
        "split": source_split,
        "study_instance_uid": str(chosen_uid),
        "competition_root_name": COMP_ROOT.name,
    },
    "models": {
        "dinov2": {
            "architecture": "DINOv2 ViT-S/14, five-role 2.5D, five-fold ensemble",
            "roles": chosen_selection["dino"],
            "role_order": DINO_ROLE_ORDER,
            "selected_role_count": int(sum(v is not None for v in chosen_selection["dino"].values())),
        },
        "vit": {
            "architecture": "Pretrained ViT-B/16 study-level model",
            "planes": chosen_selection["vit"],
            "plane_order": VIT_PLANE_ORDER,
            "selected_plane_count": int(sum(v is not None for v in chosen_selection["vit"].values())),
        },
        "ensemble": {"dinov2_weight": 0.90, "vit_weight": 0.10},
    },
    "dicom": {
        "relative_root": f"dicom/{chosen_uid}",
        "series_count": int(len(chosen_files_by_series)),
        "file_count": int(series_size_df["n_dicom_files"].sum()),
        "total_bytes": int(sum(p.stat().st_size for files in chosen_files_by_series.values() for p in files)),
        "series": [
            {
                "series_instance_uid": str(row[SERIES_COL]),
                "selected_for": str(row["selected_for"]),
                "n_files": int(row["n_dicom_files"]),
            }
            for _, row in series_size_df.iterrows()
        ],
    },
    "included_files": [
        "series_metadata.csv",
        "series_selection.csv",
        "manifest.json",
        "README.txt",
        "dicom/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm",
    ],
    "important_notes": [
        "Only the union of series selected by the final DINOv2 and ViT selection rules is included; other series from this study are intentionally omitted.",
        "The sample is from the gold_validation split and is for software demonstration only, not an independent performance evaluation.",
        "The final DINOv2 five-fold ensemble may include folds that trained on this study; do not interpret this sample prediction as out-of-fold evaluation.",
        "No study labels or pseudo-labels are included in this export.",
    ],
}

readme_text = f"""RSNA Knee Abnormality Detection — Local Demo Sample
===================================================

StudyInstanceUID: {chosen_uid}
Source split: {source_split}

This archive contains the DICOM series selected for the final two-model
inference pipeline:
- DINOv2 ViT-S/14 five-role 2.5D ensemble
- Pretrained ViT-B/16 study-level model

The archive includes the UNION of both models' selected series. A series can
be shared by both models. All DICOM files within each selected series are kept.

Contents:
- dicom/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
- series_metadata.csv: original series metadata for the selected series only
- series_selection.csv: model/role-to-series mapping
- manifest.json: export metadata and limitations

The study is from the gold_validation split. This package is for local software
demonstration only and must not be used to claim independent model performance.
No labels are included.
"""

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with zipfile.ZipFile(OUTPUT_ZIP, mode="w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    zf.writestr("demo_sample/series_metadata.csv", selected_metadata.drop(
        columns=["_plane_norm", "_vit_plane_norm", "_fluid_num", "_fat_num"], errors="ignore"
    ).to_csv(index=False))
    zf.writestr("demo_sample/series_selection.csv", selection_df.to_csv(index=False))
    zf.writestr("demo_sample/manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    zf.writestr("demo_sample/README.txt", readme_text)

    total_written = 0
    file_count = 0
    for series_uid, files in chosen_files_by_series.items():
        for file_path in files:
            arcname = Path("demo_sample") / "dicom" / str(chosen_uid) / str(series_uid) / file_path.name
            zf.write(file_path, arcname.as_posix())
            total_written += file_path.stat().st_size
            file_count += 1
            if file_count % 100 == 0:
                print(f"Added {file_count} DICOM files | {total_written / (1024 ** 2):.1f} MiB", flush=True)

# Validate ZIP structure and integrity after writing.
with zipfile.ZipFile(OUTPUT_ZIP, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity test failed at: {bad_member}")
    names = zf.namelist()
    required_names = {
        "demo_sample/series_metadata.csv",
        "demo_sample/series_selection.csv",
        "demo_sample/manifest.json",
        "demo_sample/README.txt",
    }
    if not required_names.issubset(set(names)):
        raise RuntimeError("ZIP is missing required metadata files.")

print("\nEXPORT COMPLETE")
print("Output:", OUTPUT_ZIP)
print("DICOM files:", file_count)
print("Uncompressed DICOM size (MiB):", round(total_written / (1024 ** 2), 2))
print("ZIP size (MiB):", round(OUTPUT_ZIP.stat().st_size / (1024 ** 2), 2))
display(FileLink(str(OUTPUT_ZIP)))

## 7. Cách mang dữ liệu sang local

1. Chạy notebook từ đầu đến cuối trên Kaggle.
2. Tải `demo_sample.zip` trong output của notebook về máy.
3. Giải nén vào project local, ví dụ:

```text
rsna-knee-abnormality-detection/
└── demo/
    └── input/
        └── demo_sample/
            ├── dicom/
            ├── series_metadata.csv
            ├── series_selection.csv
            ├── manifest.json
            └── README.txt
```

4. Local inference sẽ đọc `manifest.json` và `series_metadata.csv`, sau đó dùng đúng hai artifact model cuối.

### Giới hạn có chủ ý

- Chỉ export series cần cho DINOv2 + ViT; không export phần còn lại của study.
- Không export `train.csv`, ground-truth labels hoặc pseudo-labels.
- Không sửa pixel, resample hay preprocess DICOM trong notebook này. Preprocessing sẽ được thực hiện ở local bằng đúng logic tương ứng từng model.
- Nếu muốn đổi study, đặt `SAMPLE_STUDY_UID` ở Cell 1 bằng UID trong candidate table rồi chạy lại.